# Focal colours

For the tracing of synteny we need to make links of **focal** chr visible. We have to decide:

- which colour belongs to each focal chromosome
- whether only some focal chromosomes are selected
- whether grey links should be removed
- whether we are using highlight mode

This is the work hidden by `setup_focal_colors()`.

Here is breakdown how it works:

## 1. Imports

In [21]:
import matplotlib
import pandas as pd
from matplotlib.colors import to_hex

import functions
from loaders import read_colors

## 2. Small example

Species A is focal. It has three chromosomes and one link from each chromosome to species B.

In [22]:
chr_sizes = {
    "A": {"chr1": 1000, "chr2": 900, "chr3": 800},
    "B": {"chrB1": 1000, "chrB2": 900, "chrB3": 800},
}

links = pd.DataFrame({
    "chr1": ["chr1", "chr2", "chr3"],
    "gene1": ["geneA1", "geneA2", "geneA3"],
    "pos1": [100, 200, 300],
    "chr2": ["chrB1", "chrB2", "chrB3"],
    "gene2": ["geneB1", "geneB2", "geneB3"],
    "pos2": [100, 200, 300],
})

links

,chr1,gene1,pos1,chr2,gene2,pos2
0,chr1,geneA1,100,chrB1,geneB1,100
1,chr2,geneA2,200,chrB2,geneB2,200
2,chr3,geneA3,300,chrB3,geneB3,300


## 3. Colours from `colors_tsv`

The simplest case if you (user) already generated a .tsv with desired colours, then we can simply read it and append colors to chr_sizes.

`colors_tsv` controls ribbon colours. This is different from `chr_colors_tsv`, which controls the chromosome bar colours.

`read_colors()` reads the two-column file and returns chromosome names with their colors.

In [23]:
focal_colors = read_colors("tests/data/rc2.tsv")
focal_colors

{'chr1': '#ff0000', 'chr2': '#00ff00', 'chr3': '#0000ff'}

We can give this dictionary directly to the existing `assign_colors()` function.

In [45]:
links_by_pair = {
    ("A", "B"): links.copy()
}

functions.assign_colors(
    ["A", "B"],
    "A",
    focal_colors,
    links_by_pair,
    "#e3e3e3",
)

links_by_pair[("A", "B")]

,chr1,gene1,pos1,chr2,gene2,pos2,color
0,chr1,geneA1,100,chrB1,geneB1,100,#ff0000
1,chr2,geneA2,200,chrB2,geneB2,200,#00ff00
2,chr3,geneA3,300,chrB3,geneB3,300,#0000ff


## 4. Automatic colours
If you refuse to provide color input, program would need to have an option to pick the color. For that I made `auto_colors()` function.

For each chromosome it takes a colour from Matplotlib's `hsv` colour palette and returns it as a hex colour, which is the most convenient format to copy into a colour file.

>The auto assigned colours are kinda ugly tho, so it might need some improvements. 

In [30]:
color_steps = []
chromosomes = list(chr_sizes["A"].keys())
palette = matplotlib.colormaps["hsv"]
number_of_chromosomes = len(chromosomes)

for i, chromosome in enumerate(chromosomes):
    frac_for_colour = i / number_of_chromosomes # this is needed because it acepts a fraction between 0 and 1.
    hex_color = to_hex(palette(frac_for_colour))
    color_steps.append({
        "chromosome": chromosome,
        "fraction for colour": frac_for_colour,
        "hex colour": hex_color,
    })

pd.DataFrame(color_steps)

,chromosome,fraction for colour,hex colour
0,chr1,0.000000,#ff0000
1,chr2,0.333333,#08ff00
2,chr3,0.666667,#0010ff


The colour palette expects a fraction between 0 and 1. `frac_for_colour` spreads the chromosomes across that range. `to_hex()` produces a colour that can be copied directly into `colors_tsv`.

In [31]:
def auto_colors(chromosomes):
    colors = {}
    palette = matplotlib.colormaps["hsv"]
    number_of_chromosomes = len(chromosomes)

    for i, chromosome in enumerate(chromosomes):
        frac_for_colour = i / number_of_chromosomes
        colors[chromosome] = to_hex(palette(frac_for_colour))

    return colors

In [32]:
chromosomes = list(chr_sizes["A"].keys())
automatic_colors = auto_colors(chromosomes)
automatic_colors

{'chr1': '#ff0000', 'chr2': '#08ff00', 'chr3': '#0010ff'}

## 5. Selecting focal chromosomes

If the config contains:

```toml
focal_chrs = ["chr1"]
```

only `chr1` should keep a focal colour. The other links receive the default grey colour.

In [33]:
focal_chrs = ["chr1"]
selected_colors = {}

for chromosome, color in focal_colors.items():
    if chromosome in focal_chrs:
        selected_colors[chromosome] = color

selected_colors

{'chr1': '#ff0000'}

In [34]:
links_by_pair = {
    ("A", "B"): links.copy()
}

functions.assign_colors(
    ["A", "B"],
    "A",
    selected_colors,
    links_by_pair,
    "#e3e3e3",
)

colored_links = links_by_pair[("A", "B")]
colored_links

,chr1,gene1,pos1,chr2,gene2,pos2,color
0,chr1,geneA1,100,chrB1,geneB1,100,#ff0000
1,chr2,geneA2,200,chrB2,geneB2,200,#e3e3e3
2,chr3,geneA3,300,chrB3,geneB3,300,#e3e3e3


## 6. Removing grey links

A grey link does not trace back to a coloured focal chromosome.

Grey links are removed when `drop_grey = true`. They are also removed when `focal_chrs` is used, because that mode is asking for only the selected focal chromosomes and their connections.

In [44]:
drop_grey = False
remove_grey_links = drop_grey

if focal_chrs:
    remove_grey_links = True

remove_grey_links

False

In [43]:
visible_links = colored_links

if remove_grey_links:
    visible_links = colored_links[colored_links["color"] != "#e3e3e3"]

visible_links

,chr1,gene1,pos1,chr2,gene2,pos2,color
0,chr1,geneA1,100,chrB1,geneB1,100,#ff0000
1,chr2,geneA2,200,chrB2,geneB2,200,#e3e3e3
2,chr3,geneA3,300,chrB3,geneB3,300,#e3e3e3


## 7. Highlight mode

`highlight_chrs` has a different purpose from `focal_chrs`. It keeps the complete chromosome context. Links from the highlighted chromosomes are coloured and at the same time all other links stay grey.

This is why highlight mode must not remove grey links.

In [37]:
highlight_chrs = ["chr1"]
automatic_highlight_colors = auto_colors(highlight_chrs)
highlight_colors = {}

for chromosome in highlight_chrs:
    if chromosome in focal_colors:
        highlight_colors[chromosome] = focal_colors[chromosome]
    else:
        highlight_colors[chromosome] = automatic_highlight_colors[chromosome]

highlight_colors

{'chr1': '#ff0000'}

In [42]:
links_by_pair = {
    ("A", "B"): links.copy()
}

functions.assign_colors(
    ["A", "B"],
    "A",
    highlight_colors,
    links_by_pair,
    "#e3e3e3",
)

highlighted_links = links_by_pair[("A", "B")]
highlighted_links

,chr1,gene1,pos1,chr2,gene2,pos2,color
0,chr1,geneA1,100,chrB1,geneB1,100,#ff0000
1,chr2,geneA2,200,chrB2,geneB2,200,#e3e3e3
2,chr3,geneA3,300,chrB3,geneB3,300,#e3e3e3


The original engine also changes these settings in highlight mode so the grey context remains visible.

In [41]:
focal_chrs = []
remove_grey_links = False
min_links = 0
grey_alpha = 0.05

print(focal_chrs)
print(remove_grey_links)
print(min_links)
print(grey_alpha)

[]
False
0
0.05


## 8. Summary

After reviewing the steps separately, `setup_focal_colors()` is kept as a function. It returns the focal colours, the selected focal chromosomes and whether grey links should be removed.

The reason why I decided to keep `auto_colors()` as a separate function is because it performs one task and only if no color .tsv was provided, hence we should call it only if needed.